## Step 1: Install Playwright

In [ ]:
!pip install playwright
!playwright install chromium

## Step 2: Import Libraries

In [ ]:
import asyncio
from urllib.parse import urlparse
from playwright.async_api import async_playwright

print("✅ Libraries imported successfully!")

## Step 3: Configure Login Details

⚡ **EDIT THIS CONFIG - ONLY PLACE TO CHANGE!**

In [ ]:
# ====== CONFIG (ONLY CHANGE THIS) ======
CONFIG = {
    # Login details
    "LOGIN_URL": "https://rms-ui-dev-slp.supremelifeplatform.com/#/admin/login",
    "USERNAME": "ashish.admin",
    "PASSWORD": "Staging123$",
    
    # Selectors (auto-detects if fields not found)
    "USERNAME_SELECTOR": "input[type='text'], input[type='email'], input[name='email'], input[name='username']",
    "PASSWORD_SELECTOR": "input[type='password']",
    "SUBMIT_SELECTOR": "button[type='submit'], button",
    
    # Crawl settings
    "MAX_PAGES": 200,  # Maximum pages to crawl
    "OUTPUT_FILE": "extracted_links.txt",  # Save links to this file
}
# ======================================

print("📋 Configuration loaded:")
print(f"  🌐 Login URL: {CONFIG['LOGIN_URL']}")
print(f"  👤 Username: {CONFIG['USERNAME']}")
print(f"  📊 Max pages to crawl: {CONFIG['MAX_PAGES']}")
print(f"  📄 Output file: {CONFIG['OUTPUT_FILE']}")
print("\n✅ Ready to extract links!")

## Step 4: Extract All Links

This will:
1. Login to the website
2. Crawl all internal pages (BFS)
3. Extract and deduplicate all links
4. Save to text file

In [ ]:
async def extract_all_links(config):
    """Extract all internal links from authenticated website."""
    
    visited = set()
    to_visit = []
    collected = set()

    print("\n" + "="*70)
    print("🔗 Starting Link Extraction...")
    print("="*70 + "\n")

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context()
        page = await context.new_page()

        try:
            # --- LOGIN ---
            print(f"🔐 Logging in to: {config['LOGIN_URL']}")
            await page.goto(config['LOGIN_URL'], timeout=60000, wait_until='domcontentloaded')
            await asyncio.sleep(3)
            
            # Find and fill username with multiple attempts
            print("  🔍 Looking for username field...")
            try:
                await page.wait_for_selector('input', timeout=10000)
                
                username_selectors = [
                    'input[name="username"]',
                    'input[id="username"]',
                    'input[type="text"]',
                    'input[type="email"]',
                ]
                
                username_field = None
                for selector in username_selectors:
                    count = await page.locator(selector).count()
                    if count > 0:
                        username_field = page.locator(selector).first
                        await username_field.click()
                        await username_field.fill(config['USERNAME'])
                        print(f"  ✓ Username entered: {config['USERNAME']}")
                        break
                
                if not username_field:
                    raise Exception("Could not find username field")
            except Exception as e:
                print(f"  ❌ Username error: {e}")
                raise
            
            # Find and fill password with multiple attempts
            print("  🔍 Looking for password field...")
            await asyncio.sleep(1)
            
            try:
                password_selectors = [
                    'input[name="password"]',
                    'input[id="password"]',
                    'input[type="password"]',
                ]
                
                password_field = None
                for selector in password_selectors:
                    count = await page.locator(selector).count()
                    if count > 0:
                        password_field = page.locator(selector).first
                        await password_field.click()
                        await password_field.fill(config['PASSWORD'])
                        print(f"  ✓ Password entered: {'*' * len(config['PASSWORD'])}")
                        break
                
                if not password_field:
                    # Try second input as fallback
                    all_inputs = page.locator('input')
                    input_count = await all_inputs.count()
                    if input_count >= 2:
                        password_field = all_inputs.nth(1)
                        await password_field.click()
                        await password_field.fill(config['PASSWORD'])
                        print(f"  ✓ Password entered to second input field")
                    else:
                        raise Exception("Could not find password field")
            except Exception as e:
                print(f"  ❌ Password error: {e}")
                raise
            
            # Click submit button
            print("  🔍 Looking for submit button...")
            await asyncio.sleep(1)
            
            try:
                submit_selectors = [
                    'button[type="submit"]',
                    'input[type="submit"]',
                    'button:has-text("Login")',
                    'button',
                ]
                
                submit_button = None
                for selector in submit_selectors:
                    count = await page.locator(selector).count()
                    if count > 0:
                        submit_button = page.locator(selector).first
                        await submit_button.click()
                        print("  ✓ Login button clicked")
                        break
                
                if not submit_button:
                    await password_field.press('Enter')
                    print("  ✓ Pressed Enter on password field")
            except Exception as e:
                print(f"  ⚠️  Submit warning: {e}")
                await page.keyboard.press('Enter')
            
            await asyncio.sleep(3)
            
            print(f"\n✅ Login successful! Current URL: {page.url}")

            # Get base domain for filtering
            base_domain = urlparse(page.url).netloc
            to_visit.append(page.url)
            
            print(f"\n🔍 Crawling pages (max {config['MAX_PAGES']})...\n")

            # --- CRAWL ---
            while to_visit and len(visited) < config['MAX_PAGES']:
                url = to_visit.pop(0)
                
                if url in visited:
                    continue

                visited.add(url)
                print(f"  [{len(visited)}/{config['MAX_PAGES']}] Crawling: {url}")
                
                try:
                    await page.goto(url, timeout=30000, wait_until='domcontentloaded')
                    await asyncio.sleep(1)

                    # Extract all links from current page
                    links = await page.evaluate('''() => {
                        const links = Array.from(document.querySelectorAll('a[href]'));
                        return links.map(a => a.href).filter(href => href && href !== '');
                    }''')

                    # Process each link
                    for link in links:
                        if not link:
                            continue
                        
                        # Skip non-http protocols
                        if link.startswith(('mailto:', 'tel:', 'javascript:')):
                            continue

                        # Remove fragment/hash
                        link = link.split('#')[0]
                        
                        # Collect all links
                        if link:
                            collected.add(link)

                        # Queue internal links for crawling
                        if urlparse(link).netloc == base_domain and link not in visited:
                            if link not in to_visit:
                                to_visit.append(link)
                                
                except Exception as e:
                    print(f"      ⚠️  Skipped (error): {str(e)[:60]}")

            await browser.close()
            print("\n🔒 Browser closed")

        except Exception as e:
            await browser.close()
            print(f"\n❌ Error occurred: {e}")
            raise

    return sorted(collected)

# Run extraction
try:
    links = await extract_all_links(CONFIG)
    
    print("\n" + "="*70)
    print("✅ EXTRACTION COMPLETE!")
    print("="*70)
    print(f"\n📊 Total unique links found: {len(links)}")
    
    # Save to file
    with open(CONFIG['OUTPUT_FILE'], 'w', encoding='utf-8') as f:
        for link in links:
            f.write(link + '\n')
    
    print(f"💾 Links saved to: {CONFIG['OUTPUT_FILE']}")
    
    # Display first 20 links
    print("\n📋 Preview (first 20 links):")
    print("="*70)
    for i, link in enumerate(links[:20], 1):
        print(f"  {i}. {link}")
    
    if len(links) > 20:
        print(f"  ... and {len(links) - 20} more")
    
    print("\n💡 Use these links for automated screenshots or testing!")
    
except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback
    traceback.print_exc()

## 💡 How to Use the Extracted Links

The links are saved in `extracted_links.txt`. You can:

1. **Copy to screenshot automation**: Paste links into the other notebook's config
2. **Export to CSV**: Process the links for reports
3. **Use for testing**: Automated testing of all pages

### Load links programmatically:
```python
with open('extracted_links.txt', 'r') as f:
    links = [line.strip() for line in f]
```